# OpenMed Batch Dataset Processing

This notebook demonstrates scalable batch processing across directories of unstructured clinical notes using `openmed.BatchProcessor`.

Key capabilities covered:
1. Generating a batch of synthetic clinical text files.
2. Initializing `BatchProcessor` with chunking and error-handling policies.
3. Executing directory-level de-identification.

> **Privacy Invariant**: All test records are **synthetic**. No real PHI is ever processed or saved.

> **Important Note on Offline Demo Mode**:
> To enable offline execution in CI and local test suites without downloading transformer weights, this demo uses `_NoDownloadLoader`.
> **In this offline demonstration, names are matched via a fixed offline demo list, not a live NER model.**
> Production workflows with downloaded weights detect names dynamically across unconstrained vocabulary.

In [1]:
import os
import shutil
import tempfile
from typing import Any
from openmed import BatchProcessor, deidentify

# Offline mock loader for token classification on synthetic demonstration notes.
# NOTE: Names are matched via a fixed offline demo list, not a live transformer NER model.
class _NoDownloadTokenClassificationPipeline:
    tokenizer = None
    def __call__(self, inputs: Any, **_: Any) -> list[Any]:
        def _extract(text: str) -> list[dict[str, Any]]:
            spans: list[dict[str, Any]] = []
            synthetic_names = ["Alice Smith", "Bob Jones", "David Miller", "Emma Watson", "Clara Oswald"]
            for name in synthetic_names:
                idx = 0
                while True:
                    idx = text.find(name, idx)
                    if idx == -1:
                        break
                    spans.append({
                        "entity_group": "PERSON",
                        "start": idx,
                        "end": idx + len(name),
                        "score": 0.99,
                        "word": name,
                    })
                    idx += len(name)
            spans.sort(key=lambda s: s["start"])
            return spans
        if isinstance(inputs, list):
            return [_extract(t) for t in inputs]
        return _extract(inputs) if isinstance(inputs, str) else []

class _NoDownloadLoader:
    config = None
    def create_pipeline(self, *_: Any, **__: Any) -> Any:
        return _NoDownloadTokenClassificationPipeline()
    def get_max_sequence_length(self, *_: Any, **__: Any) -> None:
        return None

OFFLINE_LOADER = _NoDownloadLoader()
print("Batch processing engine configured.")

Batch processing engine configured.


## 1. Create Synthetic Input Directory

We create a temporary directory populated with synthetic clinical narratives.

In [2]:
temp_dir = tempfile.mkdtemp(prefix="openmed_gallery_batch_")
notes_dir = os.path.join(temp_dir, "input_notes")
output_dir = os.path.join(temp_dir, "redacted_notes")
os.makedirs(notes_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

SYNTHETIC_DATA = {
    "note_1.txt": "Patient Alice Smith (DOB 1980-04-12) visited clinic. Phone: 555-0101.",
    "note_2.txt": "Follow-up for Bob Jones, email bob.jones@example.test, MRN 10293847.",
    "note_3.txt": "Dr. Clara Oswald examined patient David Miller on 2026-05-18.",
    "note_4.txt": "Emma Watson admitted, contact phone +1-555-0199, SSN 000-12-3456.",
}

for name, text in SYNTHETIC_DATA.items():
    with open(os.path.join(notes_dir, name), "w", encoding="utf-8") as f:
        f.write(text)

print(f"Created {len(SYNTHETIC_DATA)} synthetic notes in input directory.")

Created 4 synthetic notes in input directory.


## 2. Execute Batch De-Identification

We configure `BatchProcessor` for directory processing and execute the workload.

In [3]:
processor = BatchProcessor(
    operation="deidentify",
    batch_size=4,
    loader=OFFLINE_LOADER,
    method="mask",
    use_safety_sweep=True,
)

batch_result = processor.process_directory(notes_dir, pattern="*.txt")
print(f"Batch Summary: {batch_result.total_items} total items, {batch_result.successful_items} succeeded, {batch_result.failed_items} failed.")

Batch Summary: 4 total items, 4 succeeded, 0 failed.


## 3. Inspect Processed Batch Outputs

We inspect the redacted output text files produced by the batch processor.

In [4]:
for item in sorted(batch_result.items, key=lambda x: x.id):
    print(f"[{item.id}] -> {item.result.deidentified_text}")

[note_1.txt] -> Patient [PERSON] (DOB [date]) visited clinic. Phone: 555-0101.
[note_2.txt] -> Follow-up for [PERSON], email [email], [medical_record_number].
[note_3.txt] -> Dr. [PERSON] examined patient [PERSON] on [date].
[note_4.txt] -> [PERSON] admitted, contact phone +1-555-0199, SSN [ssn].


## 4. Programmatic Zero-Leakage Invariant Verification

We assert that none of the target synthetic direct identifiers from the input notes survived into the redacted files.